In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch02. 딥러닝 입문 분류분석</font>
# 딥러닝
- 데이터의 규칙성 추출
- 응용 분야 : 글씨 인식, 문장 분류, 값 예측, 질병 진단, 얼굴 인식
- ML/DL의 종류
    * 지도학습 (입력, 독립 변수/타겟, 종속 변수)
        * 분류 분석 (타겟 변수가 카테고리형 변수인 분석; 다중 분류와 이진 분류)
        * 회귀 분석 (타겟 변수가 연속적인 수치형인 분석)
    * 비지도학습
        * 군집화
    * 강화학습
    
- 지도학습에서의 딥러닝 프로그래밍 방식
    1. 데이터 확보 및 생성
    2. 데이터 전처리 : 결측치 처리, 스케일 조정, 인코딩 처리 (라벨 인코딩, 원-핫 인코딩), 데이터 분리 (학습, 검증, 시험 데이터셋)
    3. 모델 구성
    4. 모델 학습 과정 설정
    5. 모델 학습시키기 (훈련, 검증 데이터셋)
    6. 모델 평가 (시험 데이터셋)
    7. 모델 저장 및 사용 (주어진 입력값을 통해 예측값 생성)

In [ ]:
from tensorflow.keras.utils import to_categorical # 분류 분석 시 원-핫 인코딩(to_categorical) 권장
from tensorflow.keras.models import Sequential, load_model # 모델 생성 및 불러오기
from tensorflow.keras.layers import Dense, Input
import pandas as pd # 원-핫 인코딩 메서드(get_dummies)
import numpy as np

## 1. 데이터셋 확보
## 2. 데이터 전처리

In [ ]:
# 학습 데이터 (훈련 데이터셋)
x_train = np.array([1,2,3,4,5,6,7,8,9]*10)
y_train = np.array([2,3,4,5,6,7,8,9,10]*10)

# 검증 데이터셋, 시험 데이터셋
x_val = np.array([1,2,3,4,5,6,7,8,9])
y_val = np.array([2,3,4,5,6,7,8,9,10])

- 인코딩 종류
    - 라벨 인코딩 : 문자열을 단순 고유 정수로 변환
    - 원-핫 인코딩 : 고유한 범주 개수만큼의 열을 새로 만들어 0과 1로 표현

In [ ]:
data = np.array(['a','b','c','b','b'])
print('원본 데이터 :', data)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# le.fit(data) - data 내 고유 범주(클래스) 목록의 정수 매핑 규칙 학습
# labeling_data = le.transform(data) - 학습한 규칙에 따라 data를 정수로 변환
labeling_data = le.fit_transform(data)
print('라벨 인코딩된 데이터 : ', labeling_data)

# to_categorical() 원-핫 인코딩 : 먼저 정수화(라벨 인코딩) 진행 필요, array 형태로 출력
# to_categorical()은 반드시 0부터 시작하는 음수가 아닌 정수(0, 1, 2, ...) 형태의 배열을 받아야함
onehot_encoding_data = to_categorical(labeling_data)
print('원-핫 인코딩된 데이터 : \n', onehot_encoding_data)

# pd.get_dummies() 원-핫 인코딩 : 라벨 인코딩 없이 원-핫 인코딩 가능, dataframe 형태로 출력
onehot_encoding_data2 = pd.get_dummies(data)
print('원-핫 인코딩된 데이터 : \n', onehot_encoding_data2)

In [ ]:
data = np.array([1,4,5,4,5])

categorical_onehot = to_categorical(data)
print('to_categorical 이용 \n', categorical_onehot)

# 인덱스와 컬럼 간의 관계가 흐트러짐
getdummies_onehot = pd.get_dummies(data)
print('get_dummies 이용 \n', getdummies_onehot)

In [ ]:
# 분류 분석을 위한 타겟(종속)변수의 원-핫 인코딩
Y_val = to_categorical(y_val)
Y_train = to_categorical(y_train)

In [ ]:
# 딥러닝 모델 훈련에 사용할 데이터
x_train.shape, Y_train.shape, x_val.shape, Y_val.shape

In [ ]:
# 출력 예시
Y_val[0]

## 3. 모델 구성

In [ ]:
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(units=38, activation='sigmoid')) # 활성화 함수 : ReLU, elu, Softmax, Sigmoid, tanh(x), Binary step, Gaussian
model.add(Dense(64, activation='elu'))
model.add(Dense(32, activation='elu'))
model.add(Dense(11, activation='softmax')) # 다중 분류에서의 출력층 활성화 함수 : Softmax
model.summary() # 학습 가능 파라미터 (Trainable params)

## 4. 모델 학습과정 설정
- 손실 함수 loss func
    - 회귀 분석 : mse, mae, rmse
    - 다중 분류 : categorical_crossentropy
    - 이진 분류 : binary_crossentropy
        
- 평가 지표 metrics
    - 분류 분석 : accuracy, recall, precision

* cross entropy error : 모델이 출력한 확률 분포(Predict)가 실제 정답의 원-핫 인코딩 분포(Target)와 얼마나 가까운지 측정

| 구분 | 이진 교차 엔트로피 (BCE) | 범주형 교차 엔트로피 (CCE) |
| :--- | :--- | :--- |
| **문제 유형** | 정답이 2개 중 하나 (이진 분류) | 정답이 3개 이상 중 하나 (다중 분류) |
| **출력층 활성화** | `Sigmoid` | `Softmax` |
| **수식** |$$
\begin{aligned}
&\text{BCE} = - \left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right]
\end{aligned}
$$|$$
\begin{aligned}
&\text{CCE} = -\sum_{k} y_k \log(\hat{y}_k)
\end{aligned}
$$
|

In [ ]:
model.compile(loss='categorical_crossentropy',optimizer='adam', metrics=['accuracy'])

## 5. 모델 학습시키기

In [ ]:
hist = model.fit(x_train, Y_train, # 학습 데이터셋 (입력 변수, 타겟 변수)
                 epochs=300, # 학습 횟수
                 batch_size=10, # 가중치 업데이트당 사용할 데이터 개수 (배치 크기)
                 # validation_split=0.2 : 학습 데이터 중 20%를 별도로 분리하여 검증용(Validation) 데이터로 사용
                 validation_data=(x_val, Y_val), # 매 Epoch마다 모델을 검증할 별도의 데이터셋 (입력, 정답)
                 verbose=1 # 0 : 학습로그 출력 없음 / 1 : 기본값 출력 / 2 : 자세한 로그 출력
                )

## 6. 모델 평가하기
- 모델 학습과정 살펴보기
- evaluate() : 학습이 완료된 모델의 최종 성능 검증(평가)

In [ ]:
hist.history.keys()

In [ ]:
# 학습과정 살펴보기
import matplotlib.pyplot as plt
fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_accuracy'], 'b', label='val_accuracy')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.25, 0.7), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.25, 0.8), ncol=1)
plt.show()

In [ ]:
# evaluate를 통한 모델 평가
score = model.evaluate(x_val, Y_val, batch_size = 3)

## 7. 모델 저장 및 사용

In [ ]:
# 모델 저장 파일 형식 : h5 / keras
model.save('model/02_deep.h5')

In [ ]:
# 저장된 모델 사용하기
model1 = load_model('model/02_deep.h5')

In [ ]:
h = model1.predict(np.array([5]))

In [ ]:
# 가장 높은 확률을 가진 정답 클래스를 최종 예측값(index)으로 추출 
h.argmax()

In [ ]:
# 최종 예측값의 확률 확인
h[0, h.argmax()]*100